In [3]:
import duckdb, pandas as pd
con = duckdb.connect("../../spectrum.duckdb")

def q(sql):
    return con.execute(sql).df()

q("SELECT * FROM customers LIMIT 10")

,customer_id,market,signup_date,plan_name,promo_price,promo_end_date,has_spectrum_mobile,has_spectrum_tv,autopay,paperless,equipment,age_band,state,list_price,competitor_fiber_available,churned,churn_date,churn_reason
0,C100000,Raleigh NC,2024-05-26,Internet Ultra 1 Gig,69.99,2025-05-21,False,False,False,True,Legacy Router,65+,NC,88.26,False,False,NaT,NaN
1,C100001,Stamford CT,2023-06-12,Internet Premier 500,49.99,2025-01-25,True,False,True,True,Advanced WiFi Router,45-54,CT,66.08,False,False,NaT,NaN
2,C100002,Dallas TX,2020-03-29,Internet Gig 2,89.99,2022-03-19,False,False,True,True,Customer-Owned,45-54,TX,127.81,True,False,NaT,NaN
3,C100003,Charlotte NC,2024-12-21,Internet Ultra 1 Gig,69.99,2025-12-16,False,False,True,True,Advanced WiFi Router,55-64,NC,97.34,True,True,2025-12-31,Service reliability
4,C100004,Columbus OH,2024-12-14,Internet Premier 500,49.99,2026-12-04,True,False,False,True,Advanced WiFi Router,55-64,OH,71.19,True,False,NaT,NaN
5,C100005,Austin TX,2019-03-31,Internet Assist,24.99,2020-03-25,False,False,True,True,Customer-Owned,35-44,TX,24.99,True,False,NaT,NaN
6,C100006,Austin TX,2020-06-20,Internet Ultra 1 Gig,69.99,2025-03-03,False,False,True,False,Advanced WiFi Router,18-24,TX,89.31,True,False,NaT,NaN
7,C100007,New York City NY,2023-10-06,Internet Ultra 1 Gig,69.99,2025-04-09,True,False,False,True,Advanced WiFi Router,25-34,NY,93.00,True,False,NaT,NaN
8,C100008,Austin TX,2022-08-02,Internet Premier 500,49.99,2023-07-28,False,False,False,True,Customer-Owned,45-54,TX,70.29,True,False,NaT,NaN
9,C100009,Dallas TX,2022-01-28,Internet Premier 500,49.99,2024-01-18,False,False,False,False,Advanced WiFi Router,55-64,TX,62.59,True,True,2025-03-03,Slow speeds


In [2]:
q("SUMMARIZE customers")


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,customer_id,VARCHAR,C100000,C119999,21093,NaN,NaN,NaN,NaN,NaN,20035,0.00
1,market,VARCHAR,ALBANY NY,Stamford CT,18,NaN,NaN,NaN,NaN,NaN,20035,0.00
2,signup_date,DATE,2019-01-01,2024-12-30,2523,2022-01-02 06:14:40.818568,NaN,2020-07-06,2022-01-01,2023-07-03,20035,0.00
3,plan_name,VARCHAR,Internet Assist,Internet Ultra 1 Gig,4,NaN,NaN,NaN,NaN,NaN,20035,0.00
4,promo_price,DOUBLE,24.99,89.99,4,57.93359870226304,16.932038673458464,49.99,49.99,69.99,20035,0.00
5,promo_end_date,DATE,2019-12-27,2026-12-20,2601,2024-01-23 06:40:24.676816,NaN,2022-08-06,2024-11-21,2025-07-09,20035,0.00
6,has_spectrum_mobile,BOOLEAN,false,true,2,NaN,NaN,NaN,NaN,NaN,20035,0.00
7,has_spectrum_tv,BOOLEAN,false,true,2,NaN,NaN,NaN,NaN,NaN,20035,0.00
8,autopay,BOOLEAN,false,true,2,NaN,NaN,NaN,NaN,NaN,20035,0.00
9,paperless,BOOLEAN,false,true,2,NaN,NaN,NaN,NaN,NaN,20035,0.00


In [4]:
q("""SELECT customer_id, COUNT(*) AS n
     FROM customers GROUP BY 1 HAVING COUNT(*) > 1""")


,customer_id,n
0,C100899,2
1,C104162,2
2,C105536,2
3,C106558,2
4,C106586,2
5,C109751,2
6,C104029,2
7,C114309,2
8,C106308,2
9,C106487,2


In [5]:
q("SELECT market, COUNT(*) AS n FROM customers GROUP BY 1 ORDER BY 1")

,market,n
0,ALBANY NY,2
1,AUSTIN TX,7
2,Albany NY,1024
3,Austin TX,1934
4,CHARLOTTE NC,9
5,COLUMBUS OH,5
6,Charlotte NC,1922
7,Columbus OH,1579
8,DALLAS TX,7
9,Dallas TX,2818


In [6]:
q("SELECT COUNT(*) AS missing_age FROM customers WHERE age_band IS NULL")

,missing_age
0,120


In [7]:
con.execute("""
CREATE OR REPLACE TABLE market_lookup AS
SELECT DISTINCT UPPER(market) AS market_key, market AS market_name
FROM customers WHERE market <> UPPER(market);

CREATE OR REPLACE TABLE customers_clean AS
SELECT DISTINCT
    c.customer_id,
    m.market_name                                   AS market,
    RIGHT(c.market, 2)                              AS state,
    CAST(c.signup_date AS DATE)                     AS signup_date,
    c.plan_name, c.promo_price, c.list_price,
    CAST(c.promo_end_date AS DATE)                  AS promo_end_date,
    c.has_spectrum_mobile, c.has_spectrum_tv, c.autopay, c.paperless,
    c.equipment,
    COALESCE(c.age_band, 'Unknown')                 AS age_band,
    c.competitor_fiber_available,
    c.churned, CAST(c.churn_date AS DATE) AS churn_date, c.churn_reason,
    (DATE '2025-12-31' - CAST(c.signup_date AS DATE)) / 365.0 AS tenure_years,
    CAST(c.promo_end_date AS DATE) BETWEEN DATE '2025-01-01' AND DATE '2025-12-31' AS promo_rolled_2025,
    c.list_price - c.promo_price                    AS price_jump
FROM customers c
JOIN market_lookup m ON UPPER(c.market) = m.market_key
""")
q("""SELECT COUNT(*) AS rows, COUNT(DISTINCT customer_id) AS customers,
            COUNT(DISTINCT market) AS markets FROM customers_clean""")

,rows,customers,markets
0,20000,20000,10


In [9]:
con.close()
